In [1]:
import yfinance as yf

In [1]:
import yfinance as yf

# Download data for a ticker
data = yf.download(
    tickers="^NSEI",        # NIFTY 50 symbol
    start="2014-01-01",     # Start date
    end="2024-12-31",       # End date
    interval="1d"           # Daily data
)

data.columns = data.columns.droplevel(1)  

print("First 5 rows:")
print(data.head())

print("\nShape (rows, columns):")
print(data.shape)

print("\nColumn info:")
print(data.info())

print("\nMissing values per column:")
print(data.isnull().sum())



data.to_csv("C:\\Financial-Time-Series_Analysis\\data\\raw\\nifty_50_data.csv", index=True) 

[*********************100%***********************]  1 of 1 completed

First 5 rows:
Price             Close         High          Low         Open  Volume
Date                                                                  
2014-01-02  6221.149902  6358.299805  6211.299805  6301.250000  158100
2014-01-03  6211.149902  6221.700195  6171.250000  6194.549805  139000
2014-01-06  6191.450195  6224.700195  6170.250000  6220.850098  118300
2014-01-07  6162.250000  6221.500000  6144.750000  6203.899902  138600
2014-01-08  6174.600098  6192.100098  6160.350098  6178.049805  146900

Shape (rows, columns):
(2698, 5)

Column info:
<class 'pandas.DataFrame'>
DatetimeIndex: 2698 entries, 2014-01-02 to 2024-12-30
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   2698 non-null   float64
 1   High    2698 non-null   float64
 2   Low     2698 non-null   float64
 3   Open    2698 non-null   float64
 4   Volume  2698 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 126.5 KB
None

Missing val

In [3]:
import pandas_market_calendars as mcal

# Get the NSE calendar
nse = mcal.get_calendar('NSE')

# Get valid trading days for your date range
schedule = nse.schedule(start_date='2014-01-01', end_date='2024-12-30')

# This gives you the actual trading days
trading_days = schedule.index

print(f"Number of official NSE trading days: {len(trading_days)}")
print(trading_days[:10])

Number of official NSE trading days: 2709
DatetimeIndex(['2014-01-01', '2014-01-02', '2014-01-03', '2014-01-06',
               '2014-01-07', '2014-01-08', '2014-01-09', '2014-01-10',
               '2014-01-13', '2014-01-14'],
              dtype='datetime64[us]', freq=None)


In [4]:
# Get the dates from your actual data
your_dates = data.index

# Convert both to the same format for comparison (just dates, no time)
official_dates = trading_days.normalize()
your_dates = your_dates.normalize()

# Find dates that are in official calendar but missing from your data
missing_dates = official_dates.difference(your_dates)

print("Missing dates:")
print(missing_dates)

Missing dates:
DatetimeIndex(['2014-01-01', '2014-02-17', '2014-10-23', '2015-01-01',
               '2015-04-15', '2015-11-11', '2016-01-01', '2016-08-12',
               '2018-01-01', '2019-01-01', '2019-02-13', '2019-03-29',
               '2020-05-25', '2024-01-22', '2024-11-20'],
              dtype='datetime64[us]', freq=None)


In [9]:
import pandas as pd

# Reload from raw file
data = pd.read_csv("C:\\Financial-Time-Series_Analysis\\data\\raw\\nifty_50_data.csv", 
                   index_col='Date', 
                   parse_dates=True)

os.makedirs("C:\\Financial-Time-Series_Analysis\\data\\processed", exist_ok=True)
data.to_csv("C:\\Financial-Time-Series_Analysis\\data\\processed\\nifty50_processed.csv")
print("Processed data saved")

Processed data saved


In [16]:
import pandas as pd
quality_log = pd.DataFrame({
    'Date': [
        '2019-02-13', 
        '2019-03-29',
        '2014-01-01', '2015-01-01', '2016-01-01', '2018-01-01', '2019-01-01',
        '2014-02-17', '2014-10-23', '2015-04-15', 
        '2015-11-11', '2016-08-12', '2020-05-25',
        '2024-01-22', '2024-11-20'
    ],
    'Issue_Type': [
        'Genuine Data Gap',
        'Genuine Data Gap',
        'Calendar Library Quirk', 'Calendar Library Quirk', 
        'Calendar Library Quirk', 'Calendar Library Quirk',
        'Calendar Library Quirk',
        'NSE Holiday', 'NSE Holiday', 'NSE Holiday',
        'NSE Holiday', 'NSE Holiday', 'NSE Holiday',
        'NSE Holiday', 'NSE Holiday'
    ],
    'Reason': [
        'NSE was open, yfinance returned no data',
        'NSE was open, yfinance returned no data',
        "New Year's Day - library incorrectly marks as trading day",
        "New Year's Day - library incorrectly marks as trading day",
        "New Year's Day - library incorrectly marks as trading day",
        "New Year's Day - library incorrectly marks as trading day",
        "New Year's Day - library incorrectly marks as trading day",
        'Mahashivratri',
        'Diwali - Laxmi Pujan',
        'Dr. Ambedkar Jayanti / Good Friday',
        'Diwali',
        'Eid al-Adha (Bakrid)',
        'Eid al-Fitr',
        'Ram Mandir Consecration - special one-off closure',
        'Gurunanak Jayanti'
    ]
})
zero_volume_log = pd.DataFrame({
    'Date': zero_volume.index.strftime('%Y-%m-%d'),
    'Issue_Type': 'Zero Volume',
    'Reason': 'yfinance returned 0 volume for index data - known data quality issue for ^NSEI'
})

# Append to existing log
full_log = pd.concat([quality_log, zero_volume_log], ignore_index=True)
full_log.to_csv("C:\\Financial-Time-Series_Analysis\\data\\logs\\data_quality_issues.csv", index=False)
print(f"Total data quality issues logged: {len(full_log)}")
print(full_log)

Total data quality issues logged: 43
          Date              Issue_Type  \
0   2019-02-13        Genuine Data Gap   
1   2019-03-29        Genuine Data Gap   
2   2014-01-01  Calendar Library Quirk   
3   2015-01-01  Calendar Library Quirk   
4   2016-01-01  Calendar Library Quirk   
5   2018-01-01  Calendar Library Quirk   
6   2019-01-01  Calendar Library Quirk   
7   2014-02-17             NSE Holiday   
8   2014-10-23             NSE Holiday   
9   2015-04-15             NSE Holiday   
10  2015-11-11             NSE Holiday   
11  2016-08-12             NSE Holiday   
12  2020-05-25             NSE Holiday   
13  2024-01-22             NSE Holiday   
14  2024-11-20             NSE Holiday   
15  2017-04-17             Zero Volume   
16  2018-04-27             Zero Volume   
17  2018-05-21             Zero Volume   
18  2018-07-06             Zero Volume   
19  2018-07-13             Zero Volume   
20  2018-08-10             Zero Volume   
21  2018-09-11             Zero Volume 

In [ ]:
# Get the dates from your actual data
your_dates = data.index

# Convert both to the same format for comparison (just dates, no time)
official_dates = trading_days.normalize()
your_dates = your_dates.normalize()

# Find dates that are in official calendar but missing from your data
missing_dates = official_dates.difference(your_dates)

print("Missing dates:")
print(missing_dates)

Missing dates:
DatetimeIndex(['2014-01-01', '2014-02-17', '2014-10-23', '2015-01-01',
               '2015-04-15', '2015-11-11', '2016-01-01', '2016-08-12',
               '2018-01-01', '2019-01-01', '2019-02-13', '2019-03-29',
               '2020-05-25', '2024-01-22', '2024-11-20'],
              dtype='datetime64[us]', freq=None)
